# DogBridge: DogSpeak + YAMNet Research Pipeline

This notebook downloads public dog vocalization data, builds a DogBridge public-data manifest, extracts YAMNet embeddings, and trains quick sanity-check classifiers.

Important framing: DogSpeak labels can help us learn canine audio structure, dog identity, breed, and sex. They do **not** directly label DogBridge owner intent labels such as `outside_bathroom`, `food_water`, or `attention`.

Sources:
- Hugging Face Hub supports `snapshot_download(..., repo_type="dataset")` for downloading dataset repositories.
- TensorFlow Hub YAMNet returns class scores, embeddings, and spectrogram values from mono waveform input.


## 0. Setup

Run the setup cell first in Colab or locally. It finds the repo from the current notebook location, a mounted Google Drive checkout, or a fresh Colab clone.

In Colab, `USE_GOOGLE_DRIVE = True` keeps the Hugging Face DogSpeak download, manifest, and YAMNet embeddings in Google Drive so runtime crashes do not force you to start from zero. The repo can still be cloned under `/content`; the large data lives in Drive.

The backend API does not need TensorFlow; this notebook is a research workspace for DogSpeak + YAMNet experiments.


In [ ]:
# Colab/local setup: install notebook deps, find or clone the repo, and make paths portable.
from pathlib import Path
import os
import subprocess
import sys

DOGBRIDGE_REPO_URL = os.environ.get(
    "DOGBRIDGE_REPO_URL",
    "https://github.com/Kritim-123/Human-Animal-Communication.git",
)
DOGBRIDGE_REPO_DIRNAME = os.environ.get("DOGBRIDGE_REPO_DIRNAME", "dogbridge")
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/DogBridge")

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
DRIVE_ROOT = None

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = DRIVE_PROJECT_DIR
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)


def run(command, cwd=None):
    print("$", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True)


def is_repo_root(candidate: Path) -> bool:
    return (candidate / "backend").is_dir() and (candidate / "notebooks" / "requirements.txt").is_file()


def find_repo_root(start: Path) -> Path | None:
    start = start.expanduser().resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if is_repo_root(candidate):
            return candidate
    for candidate in start.rglob("notebooks/requirements.txt") if start.exists() and start.is_dir() else []:
        repo_root = candidate.parents[1]
        if is_repo_root(repo_root):
            return repo_root
    return None


search_roots = [Path.cwd()]
if IN_COLAB:
    search_roots.extend([Path("/content") / DOGBRIDGE_REPO_DIRNAME])
    if DRIVE_ROOT is not None:
        search_roots.append(DRIVE_ROOT / DOGBRIDGE_REPO_DIRNAME)

REPO_ROOT = next((root for base in search_roots if (root := find_repo_root(base)) is not None), None)

if REPO_ROOT is None and IN_COLAB:
    clone_base = Path("/content")
    clone_dir = clone_base / DOGBRIDGE_REPO_DIRNAME
    if not clone_dir.exists():
        run(["git", "clone", DOGBRIDGE_REPO_URL, str(clone_dir)])
    REPO_ROOT = find_repo_root(clone_dir)

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not find the dogbridge repo. In Colab, set DOGBRIDGE_REPO_URL or mount Drive with "
        "a dogbridge checkout. Locally, open this notebook from anywhere inside the repo."
    )

os.chdir(REPO_ROOT)
NOTEBOOK_REQUIREMENTS = REPO_ROOT / "notebooks" / "requirements.txt"
if IN_COLAB or os.environ.get("DOGBRIDGE_INSTALL_NOTEBOOK_DEPS") == "1":
    get_ipython().run_line_magic("pip", f"install -q -r {NOTEBOOK_REQUIREMENTS}")

print("Repo root:", REPO_ROOT)
print("Drive project dir:", DRIVE_ROOT if DRIVE_ROOT is not None else "not mounted")
print("Python:", sys.version.split()[0])


In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


def is_repo_root(candidate: Path) -> bool:
    return (candidate / "backend").is_dir() and (candidate / "notebooks" / "requirements.txt").is_file()


if "REPO_ROOT" not in globals() or not is_repo_root(Path(REPO_ROOT)):
    current = Path.cwd().resolve()
    REPO_ROOT = next((candidate for candidate in [current, *current.parents] if is_repo_root(candidate)), None)
    if REPO_ROOT is None:
        raise RuntimeError("Run the setup cell first so the notebook can locate the dogbridge repo.")
    os.chdir(REPO_ROOT)

REPO_ROOT = Path(REPO_ROOT).resolve()
BACKEND_DIR = REPO_ROOT / "backend"

if "IN_COLAB" not in globals():
    IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
if "USE_GOOGLE_DRIVE" not in globals():
    USE_GOOGLE_DRIVE = False
if "DRIVE_ROOT" not in globals():
    DRIVE_ROOT = Path("/content/drive/MyDrive/DogBridge") if IN_COLAB and USE_GOOGLE_DRIVE else None

# In Colab, keep large public data and generated embeddings in Drive.
# Locally, keep the same repo-relative layout used by the backend docs.
DATA_ROOT = Path(DRIVE_ROOT) if DRIVE_ROOT is not None else BACKEND_DIR
PUBLIC_AUDIO_DIR = DATA_ROOT / "data" / "public" / "audio"
PUBLIC_MANIFEST_DIR = DATA_ROOT / "data" / "public" / "manifests"
PUBLIC_PROCESSED_DIR = DATA_ROOT / "data" / "processed" / "public"

for path in [PUBLIC_AUDIO_DIR, PUBLIC_MANIFEST_DIR, PUBLIC_PROCESSED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

DOGSPEAK_REPO_ID = "ArlingtonCL2/DogSpeak_Dataset"
DOGSPEAK_DIR = PUBLIC_AUDIO_DIR / "DogSpeak_Dataset"
DOGSPEAK_MANIFEST = PUBLIC_MANIFEST_DIR / "dogspeak_manifest.csv"
YAMNET_EMBEDDINGS = PUBLIC_PROCESSED_DIR / "dogspeak_yamnet_embeddings.parquet"

# Keep this small while developing. Set to None to process all clips.
SAMPLE_LIMIT = 250
RANDOM_STATE = 42
EMBEDDING_CHECKPOINT_EVERY = 25

print("Repo root:", REPO_ROOT)
print("Data root:", DATA_ROOT)
print("DogSpeak local dir:", DOGSPEAK_DIR)
print("YAMNet embeddings:", YAMNET_EMBEDDINGS)


## 1. Download DogSpeak From Hugging Face

This uses `huggingface_hub.snapshot_download` with `repo_type="dataset"` and saves into `DOGSPEAK_DIR`. In Colab with Google Drive enabled, that path is in Drive, so rerunning the notebook resumes/checks the existing folder instead of starting over.

If the dataset requires authentication, set `HF_TOKEN` in your environment or run `huggingface-cli login` first.


In [ ]:
from huggingface_hub import snapshot_download

DOGSPEAK_DIR.mkdir(parents=True, exist_ok=True)

existing_files = [path for path in DOGSPEAK_DIR.rglob("*") if path.is_file()]
if existing_files:
    print(f"Found {len(existing_files)} existing files in {DOGSPEAK_DIR}.")
    print("Checking Hugging Face for missing or changed files...")
else:
    print("No existing DogSpeak files found. Starting first download...")

download_path = snapshot_download(
    repo_id=DOGSPEAK_REPO_ID,
    repo_type="dataset",
    local_dir=str(DOGSPEAK_DIR),
)

print("Downloaded/cached at:", download_path)
print("Files now present:", sum(1 for path in DOGSPEAK_DIR.rglob("*") if path.is_file()))


## 2. Inspect Downloaded Files

DogSpeak repository layouts may change. This cell finds audio and metadata-like files instead of assuming an exact tree.

In [ ]:
audio_extensions = {".wav", ".m4a", ".mp3", ".flac", ".ogg"}
audio_files = sorted(path for path in DOGSPEAK_DIR.rglob("*") if path.suffix.lower() in audio_extensions)
metadata_candidates = sorted(
    path for path in DOGSPEAK_DIR.rglob("*")
    if path.suffix.lower() in {".csv", ".parquet", ".json", ".jsonl"}
)

print("Audio files:", len(audio_files))
print("Metadata candidates:")
for path in metadata_candidates[:20]:
    print(" -", path.relative_to(DOGSPEAK_DIR))

audio_files[:5]

## 3. Load Metadata Or Build Minimal Metadata

The project manifest builder expects `filename`, `breed`, `sex`, and `dog_id`. If the dataset metadata has different column names, map them here.

In [ ]:
def read_metadata(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".jsonl":
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    raise ValueError(f"Unsupported metadata file: {path}")

metadata = None
metadata_path = None
for candidate in metadata_candidates:
    try:
        frame = read_metadata(candidate)
        columns = set(frame.columns)
        if {"filename", "breed", "sex", "dog_id"}.issubset(columns):
            metadata = frame.copy()
            metadata_path = candidate
            break
    except Exception as exc:
        print("Skipping metadata candidate", candidate.name, exc)

if metadata is None:
    print("No exact metadata file found. Building minimal metadata from paths.")
    metadata = pd.DataFrame({
        "filename": [path.name for path in audio_files],
        "breed": "unknown",
        "sex": "unknown",
        "dog_id": [path.parent.name for path in audio_files],
    })
    metadata_path = DOGSPEAK_DIR / "dogbridge_minimal_metadata.csv"
    metadata.to_csv(metadata_path, index=False)

print("Metadata path:", metadata_path)
print(metadata.shape)
metadata.head()

## 4. Build DogBridge Public Manifest

This keeps DogSpeak data separate from owner-labeled DogBridge intent clips.

In [ ]:
from scripts.build_public_manifest import build_dogspeak_manifest

manifest = build_dogspeak_manifest(metadata_path, DOGSPEAK_DIR, DOGSPEAK_MANIFEST)
manifest["file_exists"] = manifest["file_path"].map(lambda value: Path(value).exists())

print("Manifest saved:", DOGSPEAK_MANIFEST)
print("Rows:", len(manifest))
print("Files found:", int(manifest["file_exists"].sum()))
manifest.head()

## 5. Load YAMNet

YAMNet expects mono waveform audio, usually resampled to 16 kHz. The model returns `(scores, embeddings, spectrogram)`.


In [ ]:
import librosa
import tensorflow as tf
import tensorflow_hub as hub

YAMNET_HANDLE = "https://tfhub.dev/google/yamnet/1"
yamnet_model = hub.load(YAMNET_HANDLE)
print("Loaded YAMNet:", YAMNET_HANDLE)


## 6. Extract YAMNet Embeddings

For each clip, we average frame-level YAMNet embeddings into one 1,024-dimensional vector. This is a simple, strong baseline feature.

This cell is resumable: it loads any existing embedding Parquet file, skips clips already embedded, and checkpoints progress every `EMBEDDING_CHECKPOINT_EVERY` clips.


In [ ]:
def extract_yamnet_embedding(audio_path: Path) -> np.ndarray:
    waveform, _ = librosa.load(audio_path, sr=16000, mono=True)
    if waveform.size == 0:
        raise ValueError(f"Empty audio: {audio_path}")
    scores, embeddings, spectrogram = yamnet_model(waveform.astype(np.float32))
    return embeddings.numpy().mean(axis=0)


usable = manifest[manifest["file_exists"]].copy()
if SAMPLE_LIMIT is not None:
    usable = usable.sample(min(SAMPLE_LIMIT, len(usable)), random_state=RANDOM_STATE)

if YAMNET_EMBEDDINGS.exists():
    embedding_frame = pd.read_parquet(YAMNET_EMBEDDINGS)
    completed_paths = set(embedding_frame["file_path"].astype(str)) if "file_path" in embedding_frame else set()
    print(f"Loaded existing embeddings: {embedding_frame.shape} from {YAMNET_EMBEDDINGS}")
else:
    embedding_frame = pd.DataFrame()
    completed_paths = set()
    print("No existing embedding file found. Starting extraction.")

remaining = usable[~usable["file_path"].astype(str).isin(completed_paths)].copy()
print("Usable clips:", len(usable))
print("Already embedded:", len(completed_paths))
print("Remaining clips:", len(remaining))

pending_rows = []
for index, row in tqdm(list(remaining.iterrows()), total=len(remaining)):
    path = Path(row["file_path"])
    try:
        embedding = extract_yamnet_embedding(path)
        pending_rows.append({
            "source_dataset": row["source_dataset"],
            "file_path": row["file_path"],
            "filename": row["filename"],
            "dog_id": row["dog_id"],
            "breed": row["breed"],
            "sex": row["sex"],
            **{f"yamnet_{i}": float(value) for i, value in enumerate(embedding)},
        })
    except Exception as exc:
        print("Skipping", path, exc)

    if pending_rows and len(pending_rows) >= EMBEDDING_CHECKPOINT_EVERY:
        checkpoint = pd.DataFrame(pending_rows)
        embedding_frame = pd.concat([embedding_frame, checkpoint], ignore_index=True)
        embedding_frame = embedding_frame.drop_duplicates(subset=["file_path"], keep="last")
        embedding_frame.to_parquet(YAMNET_EMBEDDINGS, index=False)
        print(f"Checkpoint saved: {embedding_frame.shape} -> {YAMNET_EMBEDDINGS}")
        pending_rows = []

if pending_rows:
    checkpoint = pd.DataFrame(pending_rows)
    embedding_frame = pd.concat([embedding_frame, checkpoint], ignore_index=True)
    embedding_frame = embedding_frame.drop_duplicates(subset=["file_path"], keep="last")
    embedding_frame.to_parquet(YAMNET_EMBEDDINGS, index=False)

print("Saved embeddings:", YAMNET_EMBEDDINGS)
print(embedding_frame.shape)
embedding_frame.head()


## 7. Sanity-Check Classifier

This does **not** predict intent. It checks whether the embeddings contain learnable dog/breed information. If this works, YAMNet embeddings are useful for DogBridge acoustic representation.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

if embedding_frame.empty:
    raise RuntimeError("No embeddings were extracted.")

feature_columns = [column for column in embedding_frame.columns if column.startswith("yamnet_")]

def train_sanity_classifier(target_column: str, min_count: int = 2):
    frame = embedding_frame.dropna(subset=[target_column]).copy()
    counts = frame[target_column].astype(str).value_counts()
    keep = counts[counts >= min_count].index
    frame = frame[frame[target_column].astype(str).isin(keep)]
    if frame[target_column].nunique() < 2 or len(frame) < 10:
        print(f"Not enough data for target={target_column}")
        return None
    x = frame[feature_columns]
    y = frame[target_column].astype(str)
    stratify = y if y.value_counts().min() >= 2 else None
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=0.25, random_state=RANDOM_STATE, stratify=stratify
    )
    model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight="balanced")
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    print("Target:", target_column)
    print(classification_report(y_test, pred, zero_division=0))
    return model

breed_model = train_sanity_classifier("breed")
dog_id_model = train_sanity_classifier("dog_id", min_count=3)

## 8. What This Means For DogBridge

If the sanity classifiers work, our next engineering step is to reuse this embedding extraction path for owner-labeled DogBridge clips.

Recommended next model path:

1. Extract YAMNet embeddings for DogBridge user clips.
2. Join embeddings with dog ID, owner label, outcome label, and context.
3. Train dog-specific intent classifiers.
4. Compare MFCC baseline vs YAMNet embeddings.
5. Keep returning `unknown` below confidence threshold.

Public data helps the acoustic side. Owner-confirmed data teaches possible intent.